# Reducing Production-Line Sensor Redundancy with PROC GVARCLUS

## Executive Summary

A multi-zone manufacturing line streams dozens of correlated sensor readings, many of which carry the same underlying signal. This notebook uses **PROC GVARCLUS** (variable clustering) to group the 13 process sensors into four disjoint clusters, then reads each cluster's R-squared values to choose one representative gauge per cluster — cutting the SPC monitoring footprint without losing process information. Three of the four clusters map cleanly onto physical zones (average R-squared 0.919, 0.932, and 0.955); the fourth is a single-channel cluster the procedure split off — the third zone-2 probe, `zone2_c` — which we examine rather than gloss over.

## Data Sources

All data is generated inline with `call streaminit(20260531)` and `rand()` — no external or network inputs.

| Dataset | Rows | Key Variables | Description |
|---------|------|---------------|-------------|
| `process_sensors` | 300 | `zone1_a`–`zone1_c`, `zone2_a`–`zone2_c`, `zone3_a`, `zone3_b`, `temp_in`, `temp_out`, `pressure_a`, `pressure_b`, `vibration` | Synthetic readings from a 3-zone production line. Sensors within a zone share a latent driver (high correlation); cross-zone gauges (temperatures tied to zone 1, pressures to zone 2, vibration to zone 3) are added to create realistic redundancy. The DATA step loop produces 300 cycles, one observation per cycle. |
| `clusters` (OUT=) | 13 | `Variable`, `Cluster`, `RSq_Own` | One row per input sensor: the cluster it was assigned to and its R-squared with that cluster's component. Produced by the `OUTPUT OUT=` statement. |

# Reducing Production-Line Sensor Redundancy with PROC GVARCLUS

On an instrumented production line, it is common to log dozens of sensors that measure overlapping physical phenomena — multiple thermocouples in one zone, redundant pressure transducers, vibration probes that track the same shaft. Feeding every channel into a control chart or downstream model wastes monitoring effort and inflates multicollinearity.

**Variable clustering** addresses this directly. `PROC GVARCLUS` partitions the numeric variables into *disjoint* clusters, where each cluster is summarized by the first principal component of its members. Sensors that move together land in the same cluster; the engineer can then keep one representative channel per cluster.

In this notebook we:

1. Synthesize 300 cycles of readings from a 3-zone line with deliberately redundant sensors.
2. Cluster the 13 channels with `PROC GVARCLUS` at `MAXCLUSTERS=4`.
3. Capture the cluster assignments in an output dataset and summarize them.
4. Interpret the R-squared values to pick one gauge per cluster for ongoing SPC.

## Step 1 — Generate synthetic sensor data

We simulate three process zones, each with a hidden driver (`zone1_a`, `zone2_a`, `zone3_a`). The remaining channels are noisy linear functions of their zone's driver, so they are strongly correlated within a zone but nearly independent across zones. We also tie the inlet/outlet temperatures to zone 1 and the two pressure transducers to zone 2, mimicking the cross-instrument redundancy seen on real lines. A fixed seed makes the run reproducible. The loop produces 300 cycles — one observation each — as the NOTE below confirms.

In [1]:
data process_sensors;
    call streaminit(20260531);
    do cycle = 1 to 300;
        /* Zone 1: latent driver plus two redundant probes */
        zone1_a = rand("normal", 0, 1);
        zone1_b = 0.90*zone1_a + rand("normal", 0, 0.30);
        zone1_c = 0.85*zone1_a + rand("normal", 0, 0.35);

        /* Zone 2: latent driver plus two redundant probes */
        zone2_a = rand("normal", 0, 1);
        zone2_b = 0.88*zone2_a + rand("normal", 0, 0.30);
        zone2_c = 0.82*zone2_a + rand("normal", 0, 0.40);

        /* Zone 3: latent driver plus one redundant probe */
        zone3_a = rand("normal", 0, 1);
        zone3_b = 0.90*zone3_a + rand("normal", 0, 0.30);

        /* Cross-instrument channels tied to existing drivers */
        temp_in    = 180 + 4.0*zone1_a + rand("normal", 0, 1.5);
        temp_out   = 178 + 4.0*zone1_a + rand("normal", 0, 1.6);
        pressure_a = 2.5 + 0.60*zone2_a + rand("normal", 0, 0.20);
        pressure_b = 2.5 + 0.58*zone2_a + rand("normal", 0, 0.22);
        vibration  = 0.40 + 0.30*zone3_a + rand("normal", 0, 0.10);
        output;
    end;
    drop cycle;
run;

NOTE: DATA process_sensors


NOTE: Wrote process_sensors (300 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds


## Step 2 — Cluster the sensors

We call `PROC GVARCLUS` on all 13 channels. The `VAR` statement lists the numeric sensors to cluster. We cap the partition at four clusters with `MAXCLUSTERS=4` (we expect roughly one cluster per physical zone, with a little slack). `ODS GRAPHICS ON` with `PLOTS=ALL` enables the variable-cluster dendrogram; the engine writes that SVG to the working directory rather than rendering it inline, so the structure we read below comes from the printed **Variable Cluster Assignments** table and the per-cluster eigenvalue table.

The `OUTPUT OUT=` statement writes the variable-to-cluster assignments — along with each variable's R-squared against its own cluster — to a dataset we can program against later.

In [2]:
ods graphics on;

proc gvarclus data=process_sensors maxclusters=4 plots=all;
    var zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
    output out=clusters;
run;

ods graphics off;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         300
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.974943
  zone1_b                                 1     0.908454
  zone1_c                                 1     0.893917
  zone2_a                                 2     0.977714
  zone2_b                                 2     0.920977
  zone2_c                                 4     1.000000
  zone3_a                                 3     0.975737
  zone3_b                                 3     0.942792
  temp_in                                 1     0.919453
  temp_out                                1     0.898186
  pressure_a                              2     0.915538
  pressure_b                              2     0.915769
  vibration  

NOTE: ODS Graphics is ON (width=640px, height=480px, format=SVG).
NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.
NOTE: ODS Graphics is OFF.


## Step 3 — Re-run with the SUMMARY option

Running the procedure a second time with the `SUMMARY` option keeps the same partition. `PLOTS=NONE` turns graphics off on this pass. In this environment the printed report is identical to Step 2 — the same 13-row assignment table, the same four clusters, and the same eigenvalue summary — so this cell mainly demonstrates that the `SUMMARY` and `PLOTS=NONE` options parse and run; it is not expected to add new numbers.

In [3]:
proc gvarclus data=process_sensors maxclusters=4 summary plots=none;
    var zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
run;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         300
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.974943
  zone1_b                                 1     0.908454
  zone1_c                                 1     0.893917
  zone2_a                                 2     0.977714
  zone2_b                                 2     0.920977
  zone2_c                                 4     1.000000
  zone3_a                                 3     0.975737
  zone3_b                                 3     0.942792
  temp_in                                 1     0.919453
  temp_out                                1     0.898186
  pressure_a                              2     0.915538
  pressure_b                              2     0.915769
  vibration  

NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.


## Step 4 — Inspect the output dataset

The `clusters` dataset from `OUTPUT OUT=` has one row per sensor with its assigned `Cluster` and `RSq_Own` (the squared correlation between the sensor and its cluster component). A high `RSq_Own` means the sensor is well represented by its cluster — an ideal candidate to *drop* in favor of the cluster's representative. We sort by cluster, then by R-squared, so the most representative sensor in each cluster sits at the top of its group.

In [4]:
proc sort data=clusters out=clusters_ranked;
    by Cluster descending RSq_Own;
run;

proc print data=clusters_ranked label;
    var Variable Cluster RSq_Own;
    label Variable = "Sensor Channel"
          Cluster  = "Cluster"
          RSq_Own  = "R-Square with Own Cluster";
run;


  Obs  Sensor Channel  Cluster  R-Square with Own Cluster
-----  --------------  -------  -------------------------
    1  ZONE1_A               1                   0.974943
    2  TEMP_IN               1                   0.919453
    3  ZONE1_B               1                   0.908454
    4  TEMP_OUT              1                   0.898186
    5  ZONE1_C               1                   0.893917
    6  ZONE2_A               2                   0.977714
    7  ZONE2_B               2                   0.920977
    8  PRESSURE_B            2                   0.915769
    9  PRESSURE_A            2                   0.915538
   10  ZONE3_A               3                   0.975737
   11  VIBRATION             3                   0.946787
   12  ZONE3_B               3                   0.942792
   13  ZONE2_C               4                          1



NOTE: PROC SORT data=clusters

NOTE: Read 13 rows from clusters.
NOTE: Wrote clusters_ranked (13 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=clusters_ranked

NOTE: PROC PRINT completed: 13 observations printed, 3 variables


## Interpreting the results

The partition recovers most of the physical structure of the line, with one honest caveat:

- **Cluster 1** gathers all five channels driven by the zone-1 latent signal: `zone1_a` (R²=0.975), `zone1_b` (0.908), `zone1_c` (0.894) and the inlet/outlet temperatures `temp_in` (0.919) and `temp_out` (0.898). Average R-squared 0.919.
- **Cluster 2** gathers two of the three zone-2 channels — `zone2_a` (0.978) and `zone2_b` (0.921) — together with the two pressure transducers `pressure_a` (0.916) and `pressure_b` (0.916), which we tied to the zone-2 driver. Average R-squared 0.932.
- **Cluster 3** gathers `zone3_a` (0.976), `zone3_b` (0.943) and `vibration` (0.947). Average R-squared 0.955.
- **Cluster 4** is a single channel: `zone2_c`, the third zone-2 probe, was split off on its own with R²=1.000 (a singleton is, trivially, perfectly explained by itself). `zone2_c` is the noisiest zone-2 channel by construction (`0.82*zone2_a` plus the largest residual SD, 0.40), so despite sharing the zone-2 driver the procedure judged it distinct enough from the `zone2_a`/pressure group to warrant its own cluster rather than merging it into Cluster 2.

The three multi-channel clusters each carry an average R-squared above **0.91**, confirming that a single component explains the bulk of the variation among their members — exactly the redundancy we want to collapse. The lone outlier — `zone2_c` landing in its own cluster instead of with the rest of zone 2 — is a useful reminder that variable clustering is data-driven: it splits on the weakest correlation, and with a different noise level or a higher `MAXCLUSTERS` budget the boundary can shift, so the partition is a starting point for engineering judgment, not a fixed truth.

**Action for the line.** Within each multi-channel cluster, keep the sensor with the highest `RSq_Own` (the channel most representative of its cluster) and retire or de-prioritize the rest from routine SPC charting — here `zone1_a` for Cluster 1, `zone2_a` for Cluster 2, and `zone3_a` for Cluster 3. Treat the `zone2_c` singleton as a flag to investigate rather than an automatic keep: confirm whether it carries genuinely distinct information or is simply the noisiest member of zone 2 before deciding to monitor it separately. Even with that one channel held aside, this collapses 13 monitored channels toward a four-channel monitoring plan, cutting alarm noise and multicollinearity while preserving the information content of the line.